# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kailaswadje/FlyRank-Internship_ML_Assignment_01_Week_01/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because it's the only one where the starter pipeline already gives me proof that a learned ranking meaningfully beats a plain rule, before I've written a single line of new modeling code myself — see Section 3, where I pull those numbers straight from the committed `outputs/model_report.md` and confirm them independently against the raw CSV. That combination — a transparent baseline, an honest client-holdout validation, and a measurable improvement over the baseline — is exactly the "did I beat the baseline" story a capstone paper needs in its Results section.

It also fits how I like to work: define a target, build the simplest possible baseline first, then earn any added complexity by showing it beats that baseline on a real held-out split — which is the same discipline I used validating my dissertation's benchmarking work (comparing candidate methods against each other under a fixed protocol, not just reporting one headline number).

In [1]:
import pandas as pd

pd.set_option('display.width', 120)

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print('starter dataset shape:', df.shape)

starter dataset shape: (30000, 44)


## 2. The question: decision, action, cost of a wrong call

**Question:** Given a limited number of review slots per week, which existing pages should a content reviewer look at first — because they show real evidence of decline, staleness, or missed opportunity — and what should the reviewer actually *do* about each one?

**Decision this improves:** how a content team allocates its scarce review time across a much larger pool of published pages, instead of reviewing pages randomly, by publish date, or only when someone happens to notice a problem.

**Who acts, and what they do:** a FlyRank content/SEO reviewer works down a ranked refresh queue. Depending on the reason codes attached to each page — declining with real demand behind it, stale but still earning impressions, thin content that's already attracting traffic, or a page holding a strong position but ageing out — they take one of a small set of content-level actions: **refresh** the page (update and re-publish it), **expand** it (it's working but thin, so grow it), or **monitor** it (not urgent yet, but worth watching). This is a content-refresh decision, not a metadata/CTR-listing decision — the reviewer is deciding what to do to the page itself, not how it's presented in search results.

**Cost of a wrong recommendation:**
- **False positive** (a page ranked high that isn't really a problem): the reviewer spends limited time refreshing a page that didn't need it — and that's time not spent on a page that genuinely was declining.
- **False negative** (a genuinely declining, high-value page missed by the ranking): the page keeps losing visibility silently, with no natural alert in the raw data — someone has to be pointed at it, or it's simply lost.

Because the review queue is finite, the value of this system is decided almost entirely by how good the *top* of the ranking is, not by overall accuracy — precision@K (specifically, precision@50, matching a realistic weekly review capacity) is the metric that reflects the real decision.

In [2]:
# No additional numbers needed here — the decision/action/cost framing is qualitative.
# The evidence that this decision is worth automating (rather than reviewed by gut feel)
# comes in Section 3, using real, already-computed results from the starter pipeline.

## 3. Quick look at the data (2-3 real numbers)

Two sources here, both real: numbers I compute fresh from `data/raw/content_refresh_anonymized.csv` below, and numbers already committed by the starter pipeline in `outputs/model_report.md` (generated by running `scripts/run_all.py` on the same CSV — I'm citing it, not re-deriving it, since re-running training isn't this week's task).

In [3]:
# Apply the same filters the starter pipeline uses (impressions_90d > 0, content_age_days >= 90),
# dedup by content_id
filtered = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')
print('rows after starter filters:', len(filtered))
print()

# trend_direction distribution -- this IS the current starter label's source, so worth
# seeing raw, not just as a single 'declining rate' summary stat
print('trend_direction distribution:')
print(filtered['trend_direction'].value_counts())
print()

# declining_with_demand reason code (from the lane guide's own baseline rule definitions):
# trend_direction == 'down' AND impressions_90d >= 100
declining_with_demand = filtered[(filtered['trend_direction'] == 'down') & (filtered['impressions_90d'] >= 100)]
pct = 100 * len(declining_with_demand) / len(filtered)
print(f"declining_with_demand candidates: {len(declining_with_demand)} ({pct:.1f}% of the filtered dataset)")
print("  -> a sizeable, well-defined pool: not too rare to model, not so common the rule is meaningless")

rows after starter filters: 30000

trend_direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

declining_with_demand candidates: 13152 (43.8% of the filtered dataset)
  -> a sizeable, well-defined pool: not too rare to model, not so common the rule is meaningless


**From the already-committed starter model report (`outputs/model_report.md`, generated by `scripts/run_all.py` on this same CSV, client-holdout validation):**

| Method | ROC AUC | Precision@50 |
|---|---:|---:|
| baseline rules | 0.627 | 0.240 |
| random forest | 0.750 | **0.740** |

Out of the top 50 pages the system would tell a reviewer to check first, the flat baseline gets about 12 right; the random forest gets about 37 right — roughly **3x better precision at the exact point that matters**, since review capacity is the scarce resource. That gap is measured on a real client-holdout split, not training-set performance, which is why I'm treating it as real evidence and not just an encouraging number.

## 4. Careful words: what I can and can't claim

**What this work will be able to say:** observed associations between a page's search/engagement signals and a future or current decline label; a ranked, reason-coded refresh queue that helps a reviewer with limited time decide where to look first; a measured, validated comparison between a simple rule and a learned model on a fixed, honest split — framed as decision support, not a guarantee.

**What it will never say:** that refreshing a flagged page *caused* a recovery (that requires an actual experiment, which this data can't give me); that any result reveals how Google's ranking algorithm works; that the model "predicts the future" in any strong sense — especially not from the current starter label, since `is_declining_label` is a same-window proxy derived from `trend_direction`, not a genuine future outcome. I'll treat that as a real limitation of this provisional pass, and it's exactly the kind of thing I'll revisit as I move from framing (this week) into the data contract and signal audit work ahead. The 0.740 precision@50 figure above is real and validated, but it's from a 30,000-row anonymized slice with client-holdout splitting — not yet the full ~79M-row warehouse.

In [4]:
# No additional numbers needed here -- the caveats above are qualitative and reference
# numbers already established and cited in Section 3 (baseline vs. random forest precision@50,
# and the same-window nature of the current starter label).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.